<!--
---
PURPOSE: Run the full pipeline with QC and reporting.
REQUIRES:
  - outputs/reports/session_inventory.parquet
PRODUCES:
  - outputs/reports/run_summary.parquet
  - outputs/reports/run_summary.html
  - outputs/reports/qc_checklist_session_{id}.json
WHAT_NEXT: Re-run specific notebooks as needed
---
-->

# 09 End-to-End Run and QC Checklist

**Purpose**
Run the full pipeline with QC and reporting.

**Requires**
- `outputs/reports/session_inventory.parquet`

**Produces**
- `outputs/reports/run_summary.parquet`
- `outputs/reports/run_summary.html`
- `outputs/reports/qc_checklist_session_{id}.json`

**What to run next**
- Re-run specific notebooks as needed

This is the main entry point for new users.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "09_End_to_End_Run_and_QC_Checklist.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Select sessions
Choose the sessions you want to process end-to-end.

In [ ]:
import json
import pandas as pd
from config import get_config
from io_sessions import load_sessions_csv, get_session_bundle
from reports import write_run_summary, write_artifact_registry

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]
SESSION_IDS

## Step 2: Run pipeline phases
We run each phase in sequence, skipping missing modalities.

In [ ]:
run_rows = []
for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    qc = {"session_id": session_id}

    # Phase 2: spikes
    units, spikes = bundle.load_spikes()
    qc["spikes"] = units is not None

    # Phase 2: behavior
    trials, events = bundle.load_trials_and_events()
    qc["behavior"] = trials is not None

    # Phase 2: eye
    eye = bundle.load_eye_features()
    qc["eye"] = eye is not None

    # Phase 3: video
    manifest = bundle.load_video_manifest(prefer_download=False)
    qc["video_alignment"] = manifest.get("alignment_method")

    run_rows.append(qc)

summary = pd.DataFrame(run_rows)
summary_path = write_run_summary(summary)
summary_path

## Step 3: Generate QC checklist and registry
We summarize pass/fail flags and list artifacts for quick review.

In [ ]:
import json
from config import get_config, make_provenance
import pandas as pd

cfg = get_config()

# QC checklist per session
for _, row in summary.iterrows():
    payload = {
        "timebase": "nwb_seconds",
        "provenance": make_provenance(int(row['session_id']), "nwb"),
        "qc": row.to_dict(),
    }
    out = cfg.outputs_dir / "reports" / f"qc_checklist_session_{int(row['session_id'])}.json"
    out.write_text(json.dumps(payload, indent=2))

# HTML summary
html_path = cfg.outputs_dir / "reports" / "run_summary.html"
summary.to_html(html_path, index=False)

registry = write_artifact_registry()
html_path, registry